In [1]:
# DB_FILE = "/Users/lulutsai/Documents/NTPU_class/paper/code/mobile01_clawler/ntpu_paper.sqlite"
LOG_FILE = "eb_annotations.log"
DB_FILE = "D:/NTPU_class/paper/code/mobile01_clawler/ntpu_paper.sqlite"
API_TOKEN = 'sk-a94e722134454665b8fef72ddb349f37'
API_URL = "http://localhost:3000/api/v1/chat/completions"
LIMIT_COUNT = 1300 # ← 你想一次處理幾筆資料（可自行調整）

In [3]:
import requests
import pandas as pd
import datetime
import json
import sqlite3
import traceback

# === 設定 ===
# DB_FILE = "/Users/lulutsai/Documents/NTPU_class/paper/code/mobile01_clawler/ntpu_paper.sqlite"
LOG_FILE = "eb_annotations.log"
DB_FILE = "D:/NTPU_class/paper/code/mobile01_clawler/ntpu_paper.sqlite"
API_TOKEN = 'sk-a94e722134454665b8fef72ddb349f37'
API_URL = "http://localhost:3000/api/v1/chat/completions"
LIMIT_COUNT = 1300 # ← 你想一次處理幾筆資料（可自行調整）
headers = {
    "Content-Type": "application/json",
    "Authorization": f"Bearer {API_TOKEN}"
}

# === 🔹 Log 工具 ===
def write_log(message: str, link_id=None, article_id=None, comment_id=None, status=None):
    timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    id_info = []
    if link_id:
        id_info.append(f"link_id={link_id}")
    if article_id:
        id_info.append(f"article_id={article_id}")
    if comment_id:
        id_info.append(f"comment_id={comment_id}")
    if status:
        id_info.append(f"status={status}")
    id_str = " | ".join(id_info)
    line = f"[{timestamp}] {message}"
    if id_str:
        line += f" | {id_str}"
    with open(LOG_FILE, "a", encoding="utf-8") as f:
        f.write(line + "\n")
    print(line)


# === 🔹 確保欄位存在 ===
def ensure_column(conn, table, column, definition):
    cur = conn.cursor()
    cur.execute(f"PRAGMA table_info({table});")
    columns = [c[1] for c in cur.fetchall()]
    if column not in columns:
        cur.execute(f"ALTER TABLE {table} ADD COLUMN {column} {definition};")
        conn.commit()
        write_log(f"🧱 已新增欄位 {table}.{column}")
    else:
        write_log(f"🔎 欄位 {column} 已存在，略過")


# === 🔹 建立 eb_annotations 資料表 ===
def ensure_eb_annotations_table(conn):
    cur = conn.cursor()
    cur.execute("""
    CREATE TABLE IF NOT EXISTS eb_annotations (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        link_id TEXT NOT NULL,
        article_id TEXT NOT NULL,
        comment_id TEXT,
        level INTEGER DEFAULT 1,
        created_at TEXT,
        strategy_power REAL,
        strategy_emotion REAL,
        strategy_blame REAL,
        score_fear REAL,
        score_obligation REAL,
        score_guilt REAL,
        pua_source TEXT,
        main_strategy TEXT,
        main_strategy_detail TEXT,
        confidence REAL,
        score_overall REAL,
        model_name TEXT,
        model_version TEXT,
        knowledge_base TEXT,
        FOREIGN KEY (article_id) REFERENCES articles(id),
        FOREIGN KEY (link_id) REFERENCES links(id)
    );
    """)
    conn.commit()
    write_log("✅ 確認資料表 eb_annotations 已存在")


# === 🔹 取得待分析文章 ===
def get_pending_articles(conn, limit):
    query = f"""
    SELECT id, link_id, content
    FROM articles
    WHERE content IS NOT NULL
      AND TRIM(content) != ''
      AND (pua_status IS NULL OR pua_status = 'pending')
    LIMIT {limit};
    """
    df = pd.read_sql_query(query, conn)
    write_log(f"📘 共讀取 {len(df)} 筆文章待分析（上限 {limit}）")
    return df


# === 🔹 更新文章狀態 ===
def update_article_status(conn, article_id, status, reason=None):
    cur = conn.cursor()
    cur.execute(
        "UPDATE articles SET pua_status = ?, pua_status_reason = ? WHERE id = ?;",
        (status, reason, article_id)
    )
    conn.commit()


# === 🔹 呼叫模型分析 ===
def analyze_text(text):
    data = {
        "model": "mistral:latest",
        "knowledge": ["pua_db"],
        "messages": [
            {
                "role": "system",
                "content": """你是一個以繁體中文回覆的情緒勒索分析助手。
請根據 Forward (1997) 與 陳燕諭 (2023) 對勒索策略的理論分類，
對給定文字進行情緒勒索分層評估，並輸出以下 JSON 結構：

{
  "strategy_power": 0~5,
  "strategy_emotion": 0~5,
  "strategy_blame": 0~5,
  "score_fear": 0~5,
  "score_obligation": 0~5,
  "score_guilt": 0~5,
  "pua_source": "family／partner／friend／workplace／online／self",
  "main_strategy": "power／emotion／blame／none",
  "main_strategy_detail": "更細緻的中文描述，例如『以哭訴方式威脅分手』或『混合型情緒操控』",
  "confidence": 0~1,
  "score_overall": 0~5,
  "is_eb_llm": 0 或 1：請判斷是否存在情緒勒索，若有情緒勒索為1，反之,
  "is_eb_llm_confidence": 0~1
}

定義說明：
- 0 代表該特徵完全沒有出現；
- 5 代表該特徵非常明顯；
- main_strategy 僅能是 power、emotion、blame、none 四類；
- 所有分數均為數值，請勿包含文字說明；
- 請務必只輸出純 JSON，不要任何解釋文字。
- is_eb_llm 為 0/1 的二元判定；is_eb_llm_confidence 為 0~1；
"""
            },
            {"role": "user", "content": f"請分析這段文字: {text}"}
        ]
    }

    res = requests.post(API_URL, headers=headers, json=data, timeout=90)
    res.raise_for_status()
    content = res.json()["choices"][0]["message"]["content"]

    try:
        result = json.loads(content)
    except json.JSONDecodeError:
        import re
        match = re.search(r'\{.*\}', content, re.S)
        if match:
            result = json.loads(match.group())
        else:
            raise ValueError(f"無法解析 JSON：{content}")
    return result


# === 🔹 分數檢查 ===
def validate_scores(result, link_id, article_id):
    for key in ["strategy_power", "strategy_emotion", "strategy_blame",
                "score_fear", "score_obligation", "score_guilt", "score_overall"]:
        if key in result:
            val = result[key]
            try:
                if not (0 <= float(val) <= 5):
                    write_log(f"⚠️ {key} 超出範圍 ({val})，已設為 None",
                              link_id=link_id, article_id=article_id)
                    result[key] = None
            except Exception:
                result[key] = None
    if "confidence" in result:
        try:
            if not (0 <= float(result["confidence"]) <= 1):
                write_log(f"⚠️ confidence 超出範圍 ({result['confidence']})，已設為 None",
                          link_id=link_id, article_id=article_id)
                result["confidence"] = None
        except Exception:
            result["confidence"] = None
    # is_eb_llm: 0/1
    if "is_eb_llm" in result:
        try:
            v = int(result["is_eb_llm"])
            result["is_eb_llm"] = v if v in (0, 1) else None
        except Exception:
            result["is_eb_llm"] = None

    # is_eb_llm_confidence: 0~1
    if "is_eb_llm_confidence" in result:
        try:
            v = float(result["is_eb_llm_confidence"])
            result["is_eb_llm_confidence"] = v if (0 <= v <= 1) else None
        except Exception:
            result["is_eb_llm_confidence"] = None
    return result

def calc_is_eb(result, overall_th=2.0, dim_th=2.0):
    """
    binary: 1=有情緒勒索跡象, 0=沒有
    規則可調：
    - score_overall >= overall_th => 1
    - 或 strategy / FOG 任一 >= dim_th => 1
    """
    def to_float(x):
        try:
            return float(x)
        except Exception:
            return None

    overall = to_float(result.get("score_overall"))
    dims = [
        to_float(result.get("strategy_power")),
        to_float(result.get("strategy_emotion")),
        to_float(result.get("strategy_blame")),
        to_float(result.get("score_fear")),
        to_float(result.get("score_obligation")),
        to_float(result.get("score_guilt")),
    ]

    if overall is not None and overall >= overall_th:
        return 1
    for v in dims:
        if v is not None and v >= dim_th:
            return 1
    return 0


# === 🔹 寫入分析結果 ===
def insert_analysis_result(conn, link_id, article_id, result):
    cur = conn.cursor()
    created_at = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    result = validate_scores(result, link_id, article_id)
    is_eb_rule = calc_is_eb(result, overall_th=2.0, dim_th=2.0)

    cur.execute("""
        INSERT INTO eb_annotations (
            link_id, article_id, comment_id, level, created_at,
            strategy_power, strategy_emotion, strategy_blame,
            score_fear, score_obligation, score_guilt,
            pua_source, main_strategy, main_strategy_detail,
            confidence, score_overall,
            is_eb_rule, is_eb_llm, is_eb_llm_confidence,
            model_name, model_version, knowledge_base
        ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?);
    """, (
        link_id, article_id, result.get("comment_id"),
        result.get("level", 1), created_at,
        result.get("strategy_power"),
        result.get("strategy_emotion"),
        result.get("strategy_blame"),
        result.get("score_fear"),
        result.get("score_obligation"),
        result.get("score_guilt"),
        result.get("pua_source"),
        result.get("main_strategy"),
        result.get("main_strategy_detail") or "無明確描述",
        result.get("confidence"),
        result.get("score_overall"),
        is_eb_rule,
        result.get("is_eb_llm"),
        result.get("is_eb_llm_confidence"),
        result.get("model_name", "mistral"),
        result.get("model_version", "latest"),
        result.get("knowledge_base", "pua_db")
    ))
    conn.commit()


# === 🔸 主程式 ===
def main():
    conn = sqlite3.connect(DB_FILE)
    ensure_column(conn, "articles", "pua_status", "TEXT DEFAULT 'pending'")
    ensure_column(conn, "articles", "pua_status_reason", "TEXT")
    ensure_eb_annotations_table(conn)
    ensure_column(conn, "eb_annotations", "main_strategy_detail", "TEXT")
    ensure_column(conn, "eb_annotations", "is_eb_rule", "INTEGER DEFAULT 0")
    ensure_column(conn, "eb_annotations", "is_eb_llm", "INTEGER")
    ensure_column(conn, "eb_annotations", "is_eb_llm_confidence", "REAL")


    df = get_pending_articles(conn, LIMIT_COUNT)
    if df.empty:
        write_log("⚠️ 沒有待分析資料。")
        conn.close()
        return

    for idx, row in df.iterrows():
        link_id = row["link_id"]
        article_id = row["id"]
        text = row["content"]

        try:
            result = analyze_text(text)
            insert_analysis_result(conn, link_id, article_id, result)
            update_article_status(conn, article_id, "done")
            is_eb_rule = calc_is_eb(result, overall_th=2.0, dim_th=2.0)
            write_log(
                f"✅ DONE 第 {idx+1}/{len(df)} 筆 | overall={result.get('score_overall')} "
                f"| rule={is_eb_rule} | llm={result.get('is_eb_llm')}({result.get('is_eb_llm_confidence')}) "
                f"| main={result.get('main_strategy')} | detail={result.get('main_strategy_detail')}",
                link_id=link_id, article_id=article_id, status="done"
            )
        except Exception as e:
            reason = f"{type(e).__name__}: {str(e)[:100]}"
            update_article_status(conn, article_id, "error", reason)
            write_log(f"❌ ERROR 第 {idx+1}/{len(df)} 筆 | {reason}",
                      link_id=link_id, article_id=article_id, status="error")
            traceback.print_exc()

    conn.close()
    write_log("🎉 全部完成！")


# # === 執行 ===
# if __name__ == "__main__":
#     main()

In [2]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("D:/NTPU_class/paper/code/mobile01_clawler/ntpu_paper.sqlite")

df = pd.read_sql_query("""
    SELECT article_id, COUNT(*) AS cnt
    FROM eb_evaluation
    GROUP BY article_id
    HAVING cnt > 1;
""", conn)

print(df)
print("🔍 有重複筆數：", len(df))

conn.close()


                    article_id  cnt
0     68fce9b6af1137205015fd06    9
1     68fce9f3af11377624f11eda   15
2     68fcea01af11377624f11edb    9
3     68fcea6eaf11379f9c929394    9
4     68fcea81af11379f9c929395    9
...                        ...  ...
1284  6923413cdbeaa944f0dd8399    9
1285  69234149dbeaa944f0dd839a    2
1286  69234156dbeaa944f0dd839b    4
1287  69234163dbeaa944f0dd839c    9
1288  69234171dbeaa944f0dd839d    9

[1289 rows x 2 columns]
🔍 有重複筆數： 1289


In [4]:
import re
import requests
import time
import json

LIMIT_COUNT =  500
def analyze_article_and_comment(article_text, comment_text, comment_id=None):
    data = {
        "model": "mistral:latest",
        "knowledge": ["pua_db", "healthy_communication_db", "social_commentary_advice_db"],
        "messages": [
                    {
                        "role": "system",
                        "content": """你是一個以繁體中文回覆的情緒勒索分析助手。
❗❗極度重要（請務必嚴格遵守，優先度最高）：
- 若評論中「沒有明確的操控、威脅、施壓、罪惡感誘發、恐懼引導、責任捆綁、指責、情緒綁架」，請將所有分數評為 0。
- 若評論內容屬於以下任一類型，也請評為 0：
  • 善意建議
  • 中立敘述
  • 理性討論
  • 情緒抒發但未要求對方負責
  • 一般社群語言、玩笑、生活分享
  • 批評但未包含控制或操作
- 除非評論中「明確存在」勒索策略，否則一律視為無勒索（即分數=0，main_strategy="none"）。
- 請勿因為語氣、情緒、抱怨、不滿，而推測或想像可能存在的勒索動機。
- 給出最低分（0）是最常見也是最正確的狀態。


📌 僅在出現「明確的施壓或操控語句」時，才可給 1~5 分。

分析規則：
- 本任務的分析主體是評論（comment）。
- 主文（article）僅提供上下文理解，不應被評分。
- 所有策略與分數僅針對評論內容進行判斷。
請根據 Forward (1997) 與 陳燕諭 (2023) 對勒索策略的理論分類，若評論中完全沒有操控成分，請將所有分數設為 0。
對給定文字進行情緒勒索分層評估，並輸出以下 JSON 結構：

{
"strategy_power": 0~5,
"strategy_emotion": 0~5,
"strategy_blame": 0~5,
"score_fear": 0~5,
"score_obligation": 0~5,
"score_guilt": 0~5,
"pua_source": "family／partner／friend／workplace／online／self",
"main_strategy": "power／emotion／blame／none",
"main_strategy_detail": "更細緻的中文描述，例如『以哭訴方式威脅分手』或『混合型情緒操控』",
"confidence": 0~1,
"score_overall": 0~5
}

補充說明：
- pua_source 請選擇「最符合的一個主要來源」。
- 若語意涉及多個來源，請選擇你判斷中影響最核心者。
- 若無法明確判定，請選擇 online。
                                """
                                                            },
                    {
                        "role": "user",
                        "content": f"""請分析以下內容，並將「評論(comment)」視為主要分析對象：

【主文（僅作為上下文，取300字）】
{article_text}

【評論（主要分析目標）】
{comment_text}""".strip()
                    }
                ]
    }
    print(data)

    # 補充說明是20260113加上得

    res = requests.post(API_URL, headers=headers, json=data, timeout=90)
    print(res.status_code)
    print(res.text)
    content = res.json()["choices"][0]["message"]["content"]

    # 避免模型包文字 → 正常處理
    try:
        result = json.loads(content)
    except:
        match = re.search(r'\{.*\}', content, re.S)
        result = json.loads(match.group()) if match else {}

    # 自動補上 comment_id
    if comment_id and "comment_id" not in result:
        result["comment_id"] = comment_id

    return result

def get_pending_comments(conn, limit):
    query = f"""
    SELECT 
        c.id AS comment_id,
        c.comment_text AS comment_text,
        a.id AS article_id,
        a.link_id AS link_id,
        a.content AS article_text
    FROM article_comments c
    JOIN articles a 
        ON c.article_id = a.id
    WHERE c.comment_text IS NOT NULL
      AND TRIM(c.comment_text) != ''
      AND (c.pua_status IS NULL OR c.pua_status = 'pending')
    LIMIT {limit};
    """
    df = pd.read_sql_query(query, conn)
    write_log(f"📘 共讀取 {len(df)} 筆評論待分析（上限 {limit}）")
    return df


def update_comment_status(conn, comment_id, status, reason=None):
    cur = conn.cursor()
    cur.execute(
        "UPDATE article_comments SET pua_status = ?, pua_status_reason = ? WHERE id = ?;",
        (status, reason, comment_id)
    )
    conn.commit()
    

def insert_analysis_result(conn, link_id, article_id, result):
    cur = conn.cursor()
    created_at = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    result = validate_scores(result, link_id, article_id)

    cur.execute("""
        INSERT INTO eb_evaluation (
            link_id, article_id, comment_id, level, created_at,
            strategy_power, strategy_emotion, strategy_blame,
            score_fear, score_obligation, score_guilt,
            pua_source, main_strategy, main_strategy_detail,
            confidence, score_overall,
            model_name, model_version, knowledge_base
        ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?);
    """, (
        link_id, article_id, result.get("comment_id"),
        result.get("level", 2), created_at,
        result.get("strategy_power"),
        result.get("strategy_emotion"),
        result.get("strategy_blame"),
        result.get("score_fear"),
        result.get("score_obligation"),
        result.get("score_guilt"),
        result.get("pua_source"),
        result.get("main_strategy"),
        result.get("main_strategy_detail") or "無明確描述",
        result.get("confidence"),
        result.get("score_overall"),
        result.get("model_name", "mistral"),
        result.get("model_version", "latest"),
        result.get("knowledge_base", "pua_db, healthy_communication_db, social_commentary_advice_db")
    ))
    conn.commit()


# === 🔸 主程式 ===
def main():
    conn = sqlite3.connect(DB_FILE)
    ensure_column(conn, "articles", "pua_status", "TEXT DEFAULT 'pending'")
    ensure_column(conn, "articles", "pua_status_reason", "TEXT")
    ensure_eb_evaluation_table(conn)
    ensure_column(conn, "eb_evaluation", "main_strategy_detail", "TEXT")
    # ⭐ 新增：評論表也要有狀態欄位
    ensure_column(conn, "article_comments", "pua_status", "TEXT DEFAULT 'pending'")
    ensure_column(conn, "article_comments", "pua_status_reason", "TEXT")

    df = get_pending_comments(conn, LIMIT_COUNT)
    if df.empty:
        write_log("⚠️ 沒有待分析評論。")
        conn.close()
        return

    for idx, row in df.iterrows():
        article_text = row["article_text"]
        comment_text = row["comment_text"]
        comment_id = row["comment_id"]
        article_id = row["article_id"]
        link_id = row["link_id"]

        try:
            result = analyze_article_and_comment(article_text, comment_text, comment_id)

            insert_analysis_result(conn, link_id, article_id, result)

            update_comment_status(conn, comment_id, "done")

            write_log(
                f"✅ DONE 第 {idx+1}/{len(df)} 筆 | overall={result.get('score_overall')} "
                f"| main={result.get('main_strategy')} | detail={result.get('main_strategy_detail')}",
                link_id=link_id, article_id=article_id, comment_id=comment_id, status="done"
            )

        except Exception as e:
            reason = f"{type(e).__name__}: {str(e)[:100]}"
            update_comment_status(conn, comment_id, "error", reason)

            write_log(
                f"❌ ERROR 第 {idx+1}/{len(df)} 筆 | {reason}",
                link_id=link_id, article_id=article_id, comment_id=comment_id, status="error"
            )
            traceback.print_exc()
        time.sleep(0.5)

    conn.close()
    write_log("🎉 全部完成！")


# === 執行 ===
if __name__ == "__main__":
    main()

[2026-02-03 10:20:41] 🔎 欄位 pua_status 已存在，略過
[2026-02-03 10:20:41] 🔎 欄位 pua_status_reason 已存在，略過
[2026-02-03 10:20:41] ✅ 確認資料表 eb_evaluation 已存在
[2026-02-03 10:20:41] 🔎 欄位 main_strategy_detail 已存在，略過
[2026-02-03 10:20:41] 🔎 欄位 pua_status 已存在，略過
[2026-02-03 10:20:41] 🔎 欄位 pua_status_reason 已存在，略過
[2026-02-03 10:20:41] 📘 共讀取 48 筆評論待分析（上限 500）
{'model': 'mistral:latest', 'knowledge': ['pua_db', 'healthy_communication_db', 'social_commentary_advice_db'], 'messages': [{'role': 'system', 'content': '你是一個以繁體中文回覆的情緒勒索分析助手。\n❗❗極度重要（請務必嚴格遵守，優先度最高）：\n- 若評論中「沒有明確的操控、威脅、施壓、罪惡感誘發、恐懼引導、責任捆綁、指責、情緒綁架」，請將所有分數評為 0。\n- 若評論內容屬於以下任一類型，也請評為 0：\n  • 善意建議\n  • 中立敘述\n  • 理性討論\n  • 情緒抒發但未要求對方負責\n  • 一般社群語言、玩笑、生活分享\n  • 批評但未包含控制或操作\n- 除非評論中「明確存在」勒索策略，否則一律視為無勒索（即分數=0，main_strategy="none"）。\n- 請勿因為語氣、情緒、抱怨、不滿，而推測或想像可能存在的勒索動機。\n- 給出最低分（0）是最常見也是最正確的狀態。\n\n\n📌 僅在出現「明確的施壓或操控語句」時，才可給 1~5 分。\n\n分析規則：\n- 本任務的分析主體是評論（comment）。\n- 主文（article）僅提供上下文理解，不應被評分。\n- 所有策略與分數僅針對評論內容進行判斷。\n請根據 Forward (1997) 與 陳燕諭 (2023) 對勒索策略的理論分類，

In [24]:
import sqlite3
import json
import pandas as pd

DB_FILE = "D:/NTPU_class/paper/code/mobile01_clawler/ntpu_paper.sqlite"

def get_comment_and_scores_as_dict(comment_id):
    conn = sqlite3.connect(DB_FILE)

    # 📝 抓評論
    df_comment = pd.read_sql_query("""
        SELECT 
            id AS comment_id,
            comment_text,
            article_id,
            link_id
        FROM article_comments
        WHERE id = ?
    """, conn, params=(comment_id,))

    if df_comment.empty:
        conn.close()
        return {"error": f"comment_id {comment_id} 不存在"}

    comment = df_comment.to_dict(orient="records")[0]

    # 📊 抓評分
    df_scores = pd.read_sql_query("""
        SELECT
            id AS eval_id,
            created_at,
            strategy_power,
            strategy_emotion,
            strategy_blame,
            score_fear,
            score_obligation,
            score_guilt,
            pua_source,
            main_strategy,
            main_strategy_detail,
            confidence,
            score_overall,
            model_name,
            model_version,
            knowledge_base
        FROM eb_evaluation
        WHERE comment_id = ?
        ORDER BY created_at ASC
    """, conn, params=(comment_id,))

    conn.close()

    scores = df_scores.to_dict(orient="records")

    return {
        "comment": comment,
        "evaluations": scores
    }


# 🧪 測試
if __name__ == "__main__":
    cid = "6923bf32af113796ec890f8b"   # ← 改成你的 comment_id
    data = get_comment_and_scores_as_dict(cid)
    print(json.dumps(data, ensure_ascii=False, indent=2))

{
  "comment": {
    "comment_id": "6923bf32af113796ec890f8b",
    "comment_text": "我覺得既然有發現問題，可以去聽聽看心理諮商師的說法，或許會找到解法也不一定～如果擔心費用現在政府也有補助方案可以詢問看看！",
    "article_id": "68fce9b6af1137205015fd06",
    "link_id": "68f87edcaf1137700cb26e94"
  },
  "evaluations": [
    {
      "eval_id": 1410,
      "created_at": "2025-12-03 10:20:20",
      "strategy_power": 0.0,
      "strategy_emotion": 4.0,
      "strategy_blame": 3.0,
      "score_fear": 2.0,
      "score_obligation": 1.0,
      "score_guilt": 1.0,
      "pua_source": "family",
      "main_strategy": "emotion",
      "main_strategy_detail": "情緒操控，使用情感哀求、指責、壓力語言等方式來勒索",
      "confidence": 1.0,
      "score_overall": 3.0,
      "model_name": "mistral",
      "model_version": "latest",
      "knowledge_base": "pua_db, healthy_communication_db, social_commentary_advice_db"
    }
  ]
}


### 檢查多選的欄位抓出來讓他們重跑

In [4]:
import sqlite3
from datetime import datetime

ALLOWED = ("emotion","power","blame","obligation","information","none")

def mark_dirty_comments_pending(conn: sqlite3.Connection) -> int:
    cur = conn.cursor()

    # 1) 找出 eb_evaluation 裡 main_strategy 髒的 comment_id
    cur.execute(f"""
        SELECT DISTINCT comment_id
        FROM eb_evaluation
        WHERE comment_id IS NOT NULL
          AND (
                main_strategy IS NULL
             OR TRIM(main_strategy) = ''
             OR main_strategy LIKE '%/%'
             OR main_strategy NOT IN {ALLOWED}
          )
    """)
    dirty_comment_ids = [r[0] for r in cur.fetchall()]
    print(dirty_comment_ids)
    print(len(dirty_comment_ids))

    if not dirty_comment_ids:
        return 0

    # 2) 把這些 comment 改回 pending
    placeholders = ",".join(["?"] * len(dirty_comment_ids))
    cur.execute(
        f"""
        UPDATE article_comments
        SET pua_status = 'pending',
            pua_status_reason = 'rerun_dirty_main_strategy'
        WHERE id IN ({placeholders})
        """,
        dirty_comment_ids
    )

    conn.commit()
    return cur.rowcount

conn = sqlite3.connect("D:/NTPU_class/paper/code/mobile01_clawler/ntpu_paper.sqlite")
dirty_count = mark_dirty_comments_pending(conn)

['6923bf32af113796ec890fd0', '6923bf32af113796ec890ff6', '6923bf32af113796ec890ffd', '6923bf32af113796ec89100e', '6923bf32af113796ec891023', '6923bf32af113796ec89102e', '6923bf32af113796ec8910b6', '6923bf32af113796ec891149', '6923bf32af113796ec89114d', '6923bf32af113796ec891155', '6923bf32af113796ec89115f', '6923bf32af113796ec89116d', '6923bf32af113796ec89116f', '6923bf32af113796ec891170', '6923bf32af113796ec891171', '6923bf32af113796ec891172', '6923bf32af113796ec891173', '6923bf32af113796ec891174', '6923bf32af113796ec891175', '6923bf32af113796ec891176', '6923bf32af113796ec891182', '6923bf32af113796ec891193', '6923bf32af113796ec891197', '6923bf32af113796ec8911f2', '6923bf32af113796ec8911fc', '6923bf32af113796ec8911ff', '6923bf32af113796ec891217', '6923bf32af113796ec891242', '6923bf32af113796ec891249', '6923bf32af113796ec891253', '6923bf32af113796ec89125b', '6923bf32af113796ec89129e', '6923bf32af113796ec8912a3', '6923bf32af113796ec8912c9', '6923bf32af113796ec8912d6', '6923bf32af113796ec

### 文章重跑（安全版本：新表 + 精確時間）

這一段只重跑文章，不動既有 `eb_annotations` 與 `articles.pua_status`。

重點：
- 新增獨立資料表 `eb_article_inference_timing`
- 保留原始資料表，避免覆蓋舊結果
- 每篇記錄 `llm_response_time_sec`、`db_write_time_sec`、`processing_time_sec`
- 用 `run_tag` 區分不同批次
- 預設只重跑「論文文章資料」，不跑評論


In [6]:
import sqlite3
import pandas as pd
import requests
import json
import re
import time
import datetime
import traceback
from pathlib import Path

# === 重跑設定 ===
DB_FILE = "D:/NTPU_class/paper/code/mobile01_clawler/ntpu_paper.sqlite"
ARTICLE_TIMING_LOG_FILE = "eb_article_inference_timing.log"
RUN_TAG = 'article_rerun_20260609_111948'
LIMIT_COUNT = 2000
ONLY_ARTICLES_USED_IN_THESIS = True
MODEL_NAME = "mistral:latest"
KNOWLEDGE_BASE = "pua_db"
REQUEST_TIMEOUT = 90
SLEEP_SEC = 0.5
RETRY_TIMES = 3
RETRY_BACKOFF_SEC = 2.0
SMOKE_TEST_COUNT = 5
MAX_CONSECUTIVE_ERRORS = 5
STOP_ON_PREFLIGHT_FAIL = True

headers = {
    "Content-Type": "application/json",
    "Authorization": f"Bearer {API_TOKEN}"
}


def write_article_timing_log(message: str, link_id=None, article_id=None, status=None, run_tag=None):
    timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    id_info = []
    if run_tag:
        id_info.append(f"run_tag={run_tag}")
    if link_id:
        id_info.append(f"link_id={link_id}")
    if article_id:
        id_info.append(f"article_id={article_id}")
    if status:
        id_info.append(f"status={status}")
    line = f"[{timestamp}] {message}"
    if id_info:
        line += " | " + " | ".join(id_info)
    with open(ARTICLE_TIMING_LOG_FILE, "a", encoding="utf-8") as f:
        f.write(line + "\n")
    print(line)


def ensure_eb_article_inference_timing_table(conn):
    cur = conn.cursor()
    cur.execute("""
    CREATE TABLE IF NOT EXISTS eb_article_inference_timing (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        run_tag TEXT NOT NULL,
        link_id TEXT NOT NULL,
        article_id TEXT NOT NULL,
        level INTEGER DEFAULT 1,
        created_at TEXT NOT NULL,
        completed_at TEXT,
        text_length INTEGER,
        text_preview TEXT,
        strategy_power REAL,
        strategy_emotion REAL,
        strategy_blame REAL,
        score_fear REAL,
        score_obligation REAL,
        score_guilt REAL,
        pua_source TEXT,
        main_strategy TEXT,
        main_strategy_detail TEXT,
        confidence REAL,
        score_overall REAL,
        is_eb_rule INTEGER,
        is_eb_llm INTEGER,
        is_eb_llm_confidence REAL,
        model_name TEXT,
        model_version TEXT,
        knowledge_base TEXT,
        llm_response_time_sec REAL,
        db_write_time_sec REAL,
        processing_time_sec REAL,
        sleep_time_sec REAL,
        status TEXT,
        error_type TEXT,
        error_message TEXT,
        FOREIGN KEY (article_id) REFERENCES articles(id),
        FOREIGN KEY (link_id) REFERENCES links(id)
    );
    """)
    cur.execute("""
    CREATE UNIQUE INDEX IF NOT EXISTS idx_eb_article_inference_timing_run_article
    ON eb_article_inference_timing(run_tag, article_id);
    """)
    conn.commit()
    write_article_timing_log("✅ 確認資料表 eb_article_inference_timing 已存在", run_tag=RUN_TAG)


def get_article_rerun_candidates(conn, limit, only_articles_used_in_thesis=True, run_tag=None):
    thesis_filter = ""
    if only_articles_used_in_thesis:
        thesis_filter = """
          AND EXISTS (
                SELECT 1
                FROM eb_annotations ea
                WHERE ea.article_id = a.id
                  AND (ea.comment_id IS NULL OR TRIM(ea.comment_id) = '')
          )
        """

    rerun_filter = ""
    params = []
    if run_tag:
        rerun_filter = """
          AND NOT EXISTS (
                SELECT 1
                FROM eb_article_inference_timing t
                WHERE t.article_id = a.id
                  AND t.run_tag = ?
          )
        """
        params.append(run_tag)

    query = f"""
    SELECT
        a.id AS article_id,
        a.link_id AS link_id,
        a.content AS content,
        LENGTH(a.content) AS text_length
    FROM articles a
    WHERE a.content IS NOT NULL
      AND TRIM(a.content) != ''
      {thesis_filter}
      {rerun_filter}
    ORDER BY a.id
    LIMIT {int(limit)};
    """
    df = pd.read_sql_query(query, conn, params=params)
    write_article_timing_log(
        f"📘 共讀取 {len(df)} 筆文章待重跑（上限 {limit}；only_thesis={only_articles_used_in_thesis}）",
        run_tag=RUN_TAG
    )
    return df


def analyze_text_for_timing(text):
    data = {
        "model": MODEL_NAME,
        "knowledge": [KNOWLEDGE_BASE],
        "messages": [
            {
                "role": "system",
                "content": """你是一個以繁體中文回覆的情緒勒索分析助手。
請根據 Forward (1997) 與 陳燕諭 (2023) 對勒索策略的理論分類，
對給定文字進行情緒勒索分層評估，並輸出以下 JSON 結構：

{
  "strategy_power": 0~5,
  "strategy_emotion": 0~5,
  "strategy_blame": 0~5,
  "score_fear": 0~5,
  "score_obligation": 0~5,
  "score_guilt": 0~5,
  "pua_source": "family／partner／friend／workplace／online／self",
  "main_strategy": "power／emotion／blame／none",
  "main_strategy_detail": "更細緻的中文描述，例如『以哭訴方式威脅分手』或『混合型情緒操控』",
  "confidence": 0~1,
  "score_overall": 0~5,
  "is_eb_llm": 0 或 1：請判斷是否存在情緒勒索，若有情緒勒索為1，反之,
  "is_eb_llm_confidence": 0~1
}

定義說明：
- 0 代表該特徵完全沒有出現；
- 5 代表該特徵非常明顯；
- main_strategy 僅能是 power、emotion、blame、none 四類；
- 所有分數均為數值，請勿包含文字說明；
- 請務必只輸出純 JSON，不要任何解釋文字。
- is_eb_llm 為 0/1 的二元判定；is_eb_llm_confidence 為 0~1；
"""
            },
            {"role": "user", "content": f"請分析這段文字: {text}"}
        ]
    }

    t0 = time.perf_counter()
    res = requests.post(API_URL, headers=headers, json=data, timeout=REQUEST_TIMEOUT)
    res.raise_for_status()
    content = res.json()["choices"][0]["message"]["content"]
    llm_elapsed = time.perf_counter() - t0

    try:
        result = json.loads(content)
    except json.JSONDecodeError:
        match = re.search(r'\{.*\}', content, re.S)
        if match:
            result = json.loads(match.group())
        else:
            raise ValueError(f"無法解析 JSON：{content}")

    return result, llm_elapsed


def analyze_with_retry(text, retry_times=RETRY_TIMES, backoff_sec=RETRY_BACKOFF_SEC):
    last_err = None
    for attempt in range(1, retry_times + 1):
        try:
            return analyze_text_for_timing(text), attempt
        except Exception as err:
            last_err = err
            if attempt < retry_times:
                sleep_sec = backoff_sec * attempt
                write_article_timing_log(
                    f"⚠️ API 失敗，第 {attempt}/{retry_times} 次重試前等待 {sleep_sec:.1f} 秒 | {type(err).__name__}: {str(err)[:120]}",
                    run_tag=RUN_TAG,
                    status="retry"
                )
                time.sleep(sleep_sec)
    raise last_err


def preflight_api_check():
    probe_text = "這是一段短測試文字，用來確認 API 是否可正常回傳 JSON。"
    write_article_timing_log("🧪 開始 API preflight 測試", run_tag=RUN_TAG)
    try:
        (result, llm_sec), attempt = analyze_with_retry(probe_text)
        write_article_timing_log(
            f"✅ API preflight 成功 | attempt={attempt} | llm_sec={llm_sec:.3f} | main={result.get('main_strategy')}",
            run_tag=RUN_TAG,
            status="ok"
        )
        return True
    except Exception as err:
        write_article_timing_log(
            f"❌ API preflight 失敗 | {type(err).__name__}: {str(err)[:200]}",
            run_tag=RUN_TAG,
            status="error"
        )
        if STOP_ON_PREFLIGHT_FAIL:
            raise
        return False


def insert_article_timing_result(conn, run_tag, link_id, article_id, text, result, llm_sec, processing_sec, status="done"):
    result = validate_scores(result, link_id, article_id)
    is_eb_rule = calc_is_eb(result, overall_th=2.0, dim_th=2.0)

    t_db_0 = time.perf_counter()
    cur = conn.cursor()
    created_at = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    cur.execute("""
        INSERT OR REPLACE INTO eb_article_inference_timing (
            run_tag, link_id, article_id, level, created_at, completed_at,
            text_length, text_preview,
            strategy_power, strategy_emotion, strategy_blame,
            score_fear, score_obligation, score_guilt,
            pua_source, main_strategy, main_strategy_detail,
            confidence, score_overall,
            is_eb_rule, is_eb_llm, is_eb_llm_confidence,
            model_name, model_version, knowledge_base,
            llm_response_time_sec, db_write_time_sec, processing_time_sec, sleep_time_sec,
            status, error_type, error_message
        ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    """, (
        run_tag, link_id, article_id, result.get("level", 1), created_at, created_at,
        len(text) if text else 0, (text or "")[:200],
        result.get("strategy_power"), result.get("strategy_emotion"), result.get("strategy_blame"),
        result.get("score_fear"), result.get("score_obligation"), result.get("score_guilt"),
        result.get("pua_source"), result.get("main_strategy"), result.get("main_strategy_detail") or "無明確描述",
        result.get("confidence"), result.get("score_overall"),
        is_eb_rule, result.get("is_eb_llm"), result.get("is_eb_llm_confidence"),
        result.get("model_name", "mistral"), result.get("model_version", "latest"), result.get("knowledge_base", KNOWLEDGE_BASE),
        round(llm_sec, 6), None, round(processing_sec, 6), SLEEP_SEC,
        status, None, None,
    ))
    conn.commit()
    db_write_sec = time.perf_counter() - t_db_0

    cur.execute(
        "UPDATE eb_article_inference_timing SET db_write_time_sec = ?, completed_at = ? WHERE run_tag = ? AND article_id = ?",
        (round(db_write_sec, 6), datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"), run_tag, article_id)
    )
    conn.commit()
    return db_write_sec


def insert_article_timing_error(conn, run_tag, link_id, article_id, text, err, llm_sec, processing_sec):
    cur = conn.cursor()
    now_str = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    cur.execute("""
        INSERT OR REPLACE INTO eb_article_inference_timing (
            run_tag, link_id, article_id, level, created_at, completed_at,
            text_length, text_preview,
            model_name, model_version, knowledge_base,
            llm_response_time_sec, processing_time_sec, sleep_time_sec,
            status, error_type, error_message
        ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    """, (
        run_tag, link_id, article_id, 1, now_str, now_str,
        len(text) if text else 0, (text or "")[:200],
        "mistral", "latest", KNOWLEDGE_BASE,
        round(llm_sec, 6) if llm_sec is not None else None,
        round(processing_sec, 6), SLEEP_SEC,
        "error", type(err).__name__, str(err)[:500],
    ))
    conn.commit()


def rerun_article_inference_with_timing(limit=LIMIT_COUNT, run_tag=RUN_TAG, only_articles_used_in_thesis=ONLY_ARTICLES_USED_IN_THESIS):
    conn = sqlite3.connect(DB_FILE)
    ensure_eb_article_inference_timing_table(conn)

    if SMOKE_TEST_COUNT and SMOKE_TEST_COUNT > 0:
        preflight_api_check()

    df = get_article_rerun_candidates(conn, limit=limit, only_articles_used_in_thesis=only_articles_used_in_thesis, run_tag=run_tag)
    if df.empty:
        write_article_timing_log("⚠️ 沒有待重跑文章。", run_tag=run_tag)
        conn.close()
        return

    write_article_timing_log(
        f"🚀 開始文章重跑：run_tag={run_tag} | 筆數={len(df)} | timeout={REQUEST_TIMEOUT} | sleep={SLEEP_SEC} | retry={RETRY_TIMES} | smoke={SMOKE_TEST_COUNT}",
        run_tag=run_tag
    )

    consecutive_errors = 0
    smoke_phase = SMOKE_TEST_COUNT if SMOKE_TEST_COUNT > 0 else 0

    for idx, row in df.iterrows():
        article_id = row["article_id"]
        link_id = row["link_id"]
        text = row["content"]
        row_t0 = time.perf_counter()
        llm_sec = None

        try:
            (result, llm_sec), attempt = analyze_with_retry(text)
            processing_sec_before_db = time.perf_counter() - row_t0
            db_write_sec = insert_article_timing_result(
                conn=conn,
                run_tag=run_tag,
                link_id=link_id,
                article_id=article_id,
                text=text,
                result=result,
                llm_sec=llm_sec,
                processing_sec=processing_sec_before_db,
                status="done",
            )
            total_processing_sec = time.perf_counter() - row_t0
            cur = conn.cursor()
            cur.execute(
                "UPDATE eb_article_inference_timing SET processing_time_sec = ? WHERE run_tag = ? AND article_id = ?",
                (round(total_processing_sec, 6), run_tag, article_id)
            )
            conn.commit()

            consecutive_errors = 0
            phase = "smoke" if idx < smoke_phase else "full"
            write_article_timing_log(
                f"✅ DONE 第 {idx+1}/{len(df)} 筆 | phase={phase} | attempt={attempt} | llm_sec={llm_sec:.3f} | db_sec={db_write_sec:.3f} | total_sec={total_processing_sec:.3f} | overall={result.get('score_overall')} | llm={result.get('is_eb_llm')}({result.get('is_eb_llm_confidence')}) | main={result.get('main_strategy')}",
                run_tag=run_tag, link_id=link_id, article_id=article_id, status="done"
            )
        except Exception as e:
            processing_sec = time.perf_counter() - row_t0
            insert_article_timing_error(conn, run_tag, link_id, article_id, text, e, llm_sec, processing_sec)
            consecutive_errors += 1
            write_article_timing_log(
                f"❌ ERROR 第 {idx+1}/{len(df)} 筆 | consecutive_errors={consecutive_errors} | total_sec={processing_sec:.3f} | {type(e).__name__}: {str(e)[:120]}",
                run_tag=run_tag, link_id=link_id, article_id=article_id, status="error"
            )
            traceback.print_exc()
            if consecutive_errors >= MAX_CONSECUTIVE_ERRORS:
                write_article_timing_log(
                    f"🛑 連續失敗達 {MAX_CONSECUTIVE_ERRORS} 筆，停止本次重跑。",
                    run_tag=run_tag,
                    status="stop"
                )
                break

        if smoke_phase and idx + 1 == smoke_phase:
            write_article_timing_log(
                f"🧪 smoke test 已完成 {smoke_phase} 筆，若結果穩定再繼續完整批次。",
                run_tag=run_tag,
                status="smoke_done"
            )

        time.sleep(SLEEP_SEC)

    conn.close()
    write_article_timing_log("🎉 文章重跑完成！", run_tag=run_tag)


# === 執行文章重跑（先確認 smoke test 設定） ===
rerun_article_inference_with_timing()


[2026-06-09 12:00:01] ✅ 確認資料表 eb_article_inference_timing 已存在 | run_tag=article_rerun_20260609_111948
[2026-06-09 12:00:01] 🧪 開始 API preflight 測試 | run_tag=article_rerun_20260609_111948
[2026-06-09 12:00:05] ✅ API preflight 成功 | attempt=1 | llm_sec=3.568 | main=none | run_tag=article_rerun_20260609_111948 | status=ok
[2026-06-09 12:00:05] 📘 共讀取 1149 筆文章待重跑（上限 2000；only_thesis=True） | run_tag=article_rerun_20260609_111948
[2026-06-09 12:00:05] 🚀 開始文章重跑：run_tag=article_rerun_20260609_111948 | 筆數=1149 | timeout=90 | sleep=0.5 | retry=3 | smoke=5 | run_tag=article_rerun_20260609_111948
[2026-06-09 12:00:10] ✅ DONE 第 1/1149 筆 | phase=smoke | attempt=1 | llm_sec=5.186 | db_sec=0.010 | total_sec=5.204 | overall=3.6 | llm=1(0.9) | main=power | run_tag=article_rerun_20260609_111948 | link_id=68f9c472af1137024c861693 | article_id=68ff534ddbeaa973acd48b59 | status=done
[2026-06-09 12:00:14] ✅ DONE 第 2/1149 筆 | phase=smoke | attempt=1 | llm_sec=3.428 | db_sec=0.009 | total_sec=3.445 | overall=4.2 